# Vytváranie aplikácií na generovanie obrázkov (Azure OpenAI)

Veľa modelov LLM je určených okrem generovania textu aj na generovanie obrázkov z textových popisov. Obrázky ako modalita sú užitočné v oblasti medicínskych technológií, architektúry, cestovného ruchu, vývoja hier, marketingu a ďalších oblastiach. V tejto lekcii pracujeme s dnešnými modelmi **GPT Image** na Microsoft Foundry.

## Ciele učenia

Po absolvovaní tejto lekcie budete vedieť:

- Vysvetliť, čo je generovanie obrázkov a kde je užitočné.
- Pochopiť rodinu modelov `gpt-image` a ako sa líši od starších modelov DALL·E.
- Vytvoriť aplikáciu na generovanie obrázkov a upravovať obrázky.

## Čo je generovanie obrázkov?

Modely generovania obrázkov vytvárajú obrázky na základe textového podnetu. Moderné modely, ako je `gpt-image`, sa počas tréningu naučia vzťah medzi textom a obrázkami a potom postupne premenia náhodný šum na obrázok, ktorý zodpovedá vášmu popisu.

Dve dobre známe rodiny modelov obrázkov sú:

- **`gpt-image` (OpenAI)** — súčasná generácia použitá v tejto lekcii. Podporuje generovanie obrázkov z textu a úpravu obrázkov (inpainting s maskou).
- **Midjourney** — populárny model od tretej strany so svojou službou a pracovným postupom založeným na Discorde.

> Staršie modely OpenAI na obrázky — **DALL·E 2** a **DALL·E 3** — sú zastarané. DALL·E 3 už nie je dostupný pre nové nasadenia a funkcie ako `create_variation` existovali len v DALL·E 2. Pre nové aplikácie používajte modely `gpt-image`.

Na Microsoft Foundry je **`gpt-image-2`** najnovší a najvýkonnejší model obrázkov a je odporúčaný ako predvolený. Modely `gpt-image-1.5` a `gpt-image-1-mini` sú tiež všeobecne dostupné.

> **Dôležité:** modely `gpt-image` vracajú vygenerovaný obrázok vo formáte **base64** (`b64_json`), nie ako URL. Váš kód dekóduje base64 reťazec na bajty a ukladá ho — neexistuje žiadna URL na stiahnutie obrázka.


## Vytvorenie vašej prvej aplikácie na generovanie obrázkov

Čo je teda potrebné na vytvorenie aplikácie na generovanie obrázkov? Potrebujete nasledujúce knižnice:

- **python-dotenv**, dôrazne sa odporúča používať túto knižnicu na uchovávanie vašich tajomstiev v súbore *.env* mimo kódu.
- **openai**, táto knižnica sa používa na interakciu s OpenAI API.
- **pillow**, na prácu s obrázkami v Pythone.

Ak ste tak ešte neurobili, postupujte podľa pokynov na stránke [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) na vytvorenie zdroja Microsoft Foundry a modelu. Vyberte model **gpt-image-2** (najnovší model obrázkov Azure OpenAI; DALL·E je zastaraný).

1. Vytvorte súbor *.env* s nasledujúcim obsahom:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Túto informáciu nájdete v portáli Microsoft Foundry pre váš zdroj v sekcii "Deployments".



1. Zbaľte vyššie uvedené knižnice do súboru nazvaného *requirements.txt* takto:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ďalej vytvorte virtuálne prostredie a nainštalujte knižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pre Windows použite nasledujúce príkazy na vytvorenie a aktiváciu vášho virtuálneho prostredia:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Pridajte nasledujúci kód do súboru s názvom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # nakonfigurujte klienta Azure OpenAI (Microsoft Foundry).
    # Modely obrázkov vyžadujú aktuálnu verziu API — skontrolujte dokumentáciu Foundry pre verziu požadovanú vaším modelom.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Vytvorte obrázok pomocou API na generovanie obrázkov
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Zadajte text vášho promptu sem
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # napr. "gpt-image-2"
        )
        # Nastavte adresár pre uložený obrázok
        image_dir = os.path.join(os.curdir, 'images')

        # Ak adresár neexistuje, vytvorte ho
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializujte cestu k obrázku (vezmite na vedomie, že typ súboru by mal byť png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Modely gpt-image vracajú obrázok ako base64 (b64_json), nie URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Zobrazte obrázok v predvolenom prehliadači obrázkov
        image = Image.open(image_path)
        image.show()

    # zachyťte výnimky
    except BadRequestError as err:
        print(err)

    ```

Vysvetlime si tento kód:

- Najprv naimportujeme knižnice, ktoré potrebujeme, vrátane knižnice OpenAI, knižnice dotenv, modulu base64 a knižnice Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Potom načítame premenné prostredia zo súboru *.env*.

    ```python
    # importovať dotenv
    dotenv.load_dotenv()
    ```

- Následne nakonfigurujeme klienta Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Ďalej generujeme obrázok:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Sem zadajte text vášho promptu
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Modely `gpt-image` vracajú obrázok ako **base64** reťazec v `data[0].b64_json`. Dekódujeme ho do bajtov a zapíšeme do súboru — nenachádza sa tu URL na stiahnutie.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Nakoniec otvoríme obrázok a použijeme štandardný prehliadač obrázkov na jeho zobrazenie:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Viac podrobností o generovaní obrázka

Pozrime sa na parametre `images.generate`:

- **prompt** je textový prompt použitý na vygenerovanie obrázka. Tu je to "Zajko na koni, drží lízanku, na hmlistej lúke, kde rastú narcisové kvety".
- **size** je veľkosť vygenerovaného obrázka (napríklad `1024x1024`, `1536x1024`, `1024x1536` alebo `"auto"`).
- **n** je počet generovaných obrázkov. Tu generujeme jeden.
- **model** je názov nasadenia vášho obrazového modelu (napríklad `gpt-image-2`).

> Obrazové modely nevyužívajú parameter `temperature` — ten je určený pre generovanie textu. Pre väčšiu rôznorodosť volajte API znova; pre menšiu rôznorodosť urobte váš prompt konkrétnejším.

## Ďalšie možnosti generovania obrázkov

Už ste videli, ako vygenerovať obrázok niekoľkými riadkami Pythonu. Modely `gpt-image` môžu tiež **upravovať** existujúci obrázok. Poskytnutím obrázka, voliteľnej **masky** (ktorá označuje oblasť na zmenu) a promptu môžete zmeniť časť obrázka — napríklad pridať klobúk nášmu zajkovi.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# úpravy sú tiež vrátené vo formáte base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Základný obrázok obsahuje iba zajaca; konečný obrázok má zajaca s klobúkom.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
